# Deteccao de fraudes em cartoes de credito
### Analise exploratoria e comparacao de modelos

Este notebook trabalha com o dataset **Credit Card Fraud Detection**, disponivel no Kaggle. A base reune transacoes de cartoes europeus realizadas em setembro de 2013 e tem uma caracteristica central: as fraudes sao muito raras em relacao ao total de transacoes.

O objetivo e montar um fluxo de analise para esse tipo de problema, passando pela leitura dos dados, exploracao inicial, tratamento do desbalanceamento e comparacao de modelos de classificacao. Como as variaveis originais foram anonimizadas por PCA, a interpretacao direta de negocio e limitada. Por isso, a analise se concentra no comportamento estatistico das variaveis e nas metricas dos modelos.

## Dataset

A base original possui:

- 284.807 transacoes;
- 492 fraudes, cerca de 0,172% do total;
- as colunas `Time`, `Amount`, `V1` a `V28` e a variavel alvo `Class`;
- `Class = 0` para transacoes legitimas e `Class = 1` para fraudes.

Por questao de licenca, o arquivo `creditcard.csv` precisa ser baixado manualmente pelo Kaggle:
https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud

Depois do download, basta deixar o arquivo na mesma pasta do notebook. Se ele nao estiver disponivel, o codigo cria uma base sintetica com a mesma estrutura geral, apenas para permitir que o fluxo seja executado. As conclusoes sobre o problema real devem ser tiradas usando a base original.

## Etapas do notebook

1. Configuracao do ambiente
2. Carregamento dos dados
3. Exploracao inicial
4. Distribuicao das classes
5. Analise de `Time` e `Amount`
6. Correlacao entre variaveis e classe
7. Pre-processamento
8. Separacao entre treino e teste
9. Tratamento do desbalanceamento
10. Regressao Logistica como baseline
11. Random Forest
12. Comparacao dos modelos
13. Ajuste do threshold de decisao
14. Importancia das variaveis
15. Conclusoes

## 1. Configuracao do ambiente

Nesta primeira parte ficam os imports usados no notebook: manipulacao de dados, visualizacao, modelos, metricas e o `SMOTE`, quando a biblioteca `imbalanced-learn` estiver instalada.

Tambem deixo uma seed fixa em `RANDOM_STATE` para facilitar a reproducao dos resultados.

In [ ]:
# Dados
import pandas as pd
import numpy as np

# Visualizacao
import matplotlib.pyplot as plt
import seaborn as sns

# Modelos
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Metricas
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
)

# SMOTE e opcional. O restante do notebook continua funcionando sem ele.
try:
    from imblearn.over_sampling import SMOTE
    SMOTE_DISPONIVEL = True
except ImportError:
    SMOTE_DISPONIVEL = False
    print("imbalanced-learn nao esta instalado. Para usar SMOTE, rode: pip install imbalanced-learn")

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Ambiente configurado.")

## 2. Carregamento dos dados

O notebook procura pelo arquivo `creditcard.csv` na mesma pasta. Caso ele nao exista, e criada uma base sintetica com as mesmas colunas e uma proporcao de fraude proxima da base real.

A base sintetica serve apenas para testar o codigo de ponta a ponta. Para analisar o problema de fato, o ideal e baixar o arquivo original do Kaggle e executar o notebook novamente.

In [ ]:
def gerar_dataset_sintetico(n_amostras=120_000, taxa_fraude=0.00172, random_state=42):
    """
    Gera uma base sintetica com a estrutura do creditcard.csv original.
    Ela e usada apenas quando o arquivo real nao esta disponivel.
    """
    rng = np.random.RandomState(random_state)

    n_fraude = max(int(n_amostras * taxa_fraude), 50)
    n_normal = n_amostras - n_fraude

    # Transacoes legitimas.
    normal = pd.DataFrame(
        rng.normal(loc=0.0, scale=1.0, size=(n_normal, 28)),
        columns=[f"V{i}" for i in range(1, 29)],
    )
    normal["Time"] = rng.uniform(0, 172_792, size=n_normal)
    normal["Amount"] = rng.lognormal(mean=3.0, sigma=1.2, size=n_normal).round(2)
    normal["Class"] = 0

    # Fraudes simuladas com deslocamento em algumas componentes.
    fraude = pd.DataFrame(
        rng.normal(loc=0.0, scale=1.0, size=(n_fraude, 28)),
        columns=[f"V{i}" for i in range(1, 29)],
    )

    # O deslocamento abaixo cria um sinal minimo para os modelos aprenderem.
    for col, shift, scale in [("V1", -3.5, 2.0), ("V3", -4.0, 2.2),
                               ("V4", 2.8, 1.8), ("V10", -3.0, 1.7),
                               ("V14", -3.8, 2.0), ("V17", -2.5, 1.6)]:
        fraude[col] = rng.normal(loc=shift, scale=scale, size=n_fraude)

    fraude["Time"] = rng.uniform(0, 172_792, size=n_fraude)
    fraude["Amount"] = rng.lognormal(mean=2.2, sigma=1.5, size=n_fraude).round(2)
    fraude["Class"] = 1

    df_sint = pd.concat([normal, fraude], ignore_index=True)
    cols_ordenadas = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount", "Class"]
    df_sint = df_sint[cols_ordenadas]
    df_sint = df_sint.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    return df_sint


DATA_PATH = "creditcard.csv"
USANDO_DADOS_SINTETICOS = False

try:
    df = pd.read_csv(DATA_PATH)
    print(f"Dataset original carregado de '{DATA_PATH}'.")
except FileNotFoundError:
    print(f"Arquivo '{DATA_PATH}' nao encontrado.")
    print("Gerando uma base sintetica apenas para testar o fluxo do notebook...\n")
    df = gerar_dataset_sintetico(random_state=RANDOM_STATE)
    USANDO_DADOS_SINTETICOS = True
    print("Base sintetica gerada.")

print(f"\nFormato do dataset: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
df.head()

## 3. Exploracao inicial dos dados

Antes de partir para os modelos, faco uma checagem basica da base: tipos das colunas, valores nulos, duplicados e estatisticas iniciais. Essa etapa ajuda a evitar problemas simples mais adiante.

In [ ]:
print("=" * 60)
print("INFORMACOES GERAIS")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("VALORES NULOS POR COLUNA")
print("=" * 60)
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else "Nenhum valor nulo encontrado.")

print("\n" + "=" * 60)
print("LINHAS DUPLICADAS")
print("=" * 60)
print(f"Total de duplicadas: {df.duplicated().sum():,}")

In [ ]:
# Time e Amount sao as variaveis com interpretacao direta.
# As colunas V1..V28 ficam resumidas, pois sao componentes PCA anonimizadas.
print("Estatisticas de Time e Amount:")
display(df[["Time", "Amount"]].describe().T)

print("\nEstatisticas agregadas das componentes PCA (V1..V28):")
v_cols = [c for c in df.columns if c.startswith("V")]
display(df[v_cols].describe().T[["mean", "std", "min", "max"]].describe())

## 4. Desbalanceamento das classes

A distribuicao da variavel alvo e o primeiro ponto importante deste problema. Como as fraudes representam uma parcela muito pequena da base, um modelo pode ter acuracia alta mesmo sem identificar bem as fraudes.

Por isso, a avaliacao precisa olhar para metricas como precision, recall, F1-score e PR-AUC, que mostram melhor o desempenho na classe minoritaria.

In [ ]:
contagem = df["Class"].value_counts().sort_index()
percentual = df["Class"].value_counts(normalize=True).sort_index() * 100

resumo = pd.DataFrame({
    "Classe": ["Legitima (0)", "Fraude (1)"],
    "Quantidade": contagem.values,
    "Percentual (%)": percentual.values.round(4),
})
display(resumo)

razao = contagem[0] / contagem[1]
print(f"\nPara cada fraude, ha aproximadamente {razao:,.0f} transacoes legitimas.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Quantidade por classe.
sns.barplot(x=["Legitima (0)", "Fraude (1)"], y=contagem.values,
            palette=["#2ecc71", "#e74c3c"], ax=axes[0])
axes[0].set_title("Distribuicao absoluta das classes")
axes[0].set_ylabel("Quantidade de transacoes")
for i, v in enumerate(contagem.values):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom", fontweight="bold")

# Percentual por classe.
axes[1].pie(
    percentual.values,
    labels=["Legitima", "Fraude"],
    autopct="%1.3f%%",
    colors=["#2ecc71", "#e74c3c"],
    explode=(0, 0.15),
    startangle=90,
)
axes[1].set_title("Distribuicao percentual das classes")

plt.tight_layout()
plt.show()

A visualizacao deixa a proporcao entre as classes mais concreta. A classe de fraude e pequena demais para ser avaliada por acuracia simples, entao as proximas etapas usam metricas mais adequadas para classe rara.

## 5. Analise de `Time` e `Amount`

`Time` representa os segundos decorridos desde a primeira transacao da base, enquanto `Amount` representa o valor da transacao. Aqui comparo as distribuicoes dessas duas variaveis entre transacoes legitimas e fraudes.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribuicao de Amount.
sns.histplot(df[df["Class"] == 0]["Amount"], bins=50, color="#2ecc71",
             ax=axes[0, 0], kde=False)
axes[0, 0].set_title("Distribuicao de Amount - transacoes legitimas")
axes[0, 0].set_xlim(0, df["Amount"].quantile(0.99))

sns.histplot(df[df["Class"] == 1]["Amount"], bins=50, color="#e74c3c",
             ax=axes[0, 1], kde=False)
axes[0, 1].set_title("Distribuicao de Amount - fraudes")
axes[0, 1].set_xlim(0, df["Amount"].quantile(0.99))

# Distribuicao de Time.
sns.histplot(df[df["Class"] == 0]["Time"], bins=50, color="#2ecc71", ax=axes[1, 0])
axes[1, 0].set_title("Distribuicao de Time - transacoes legitimas")

sns.histplot(df[df["Class"] == 1]["Time"], bins=50, color="#e74c3c", ax=axes[1, 1])
axes[1, 1].set_title("Distribuicao de Time - fraudes")

plt.tight_layout()
plt.show()

print("Estatisticas de Amount por classe:")
display(df.groupby("Class")["Amount"].describe())

O `Amount` pode indicar diferencas uteis entre as classes, principalmente quando ha concentracao de fraudes em faixas especificas de valor. O `Time`, sozinho, tende a ser menos conclusivo, mas ainda pode contribuir quando combinado com as demais variaveis.

## 6. Correlacoes entre variaveis e classe

As variaveis `V1` a `V28` sao componentes geradas por PCA, entao a correlacao entre elas tende a ser baixa. O que mais interessa aqui e a correlacao de cada variavel com `Class`, pois isso da uma primeira nocao de quais componentes ajudam a separar fraude de transacao legitima.

In [ ]:
corr_com_classe = df.corr()["Class"].drop("Class").sort_values()

plt.figure(figsize=(8, 10))
colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in corr_com_classe.values]
plt.barh(corr_com_classe.index, corr_com_classe.values, color=colors)
plt.title("Correlacao de cada variavel com a classe de fraude")
plt.xlabel("Coeficiente de correlacao")
plt.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

print("Top 5 correlacoes positivas com fraude:")
print(corr_com_classe.sort_values(ascending=False).head(5))
print("\nTop 5 correlacoes negativas com fraude:")
print(corr_com_classe.sort_values().head(5))

In [ ]:
# O heatmap serve como uma conferencia geral das correlacoes da base.
plt.figure(figsize=(16, 13))
sns.heatmap(df.corr(), cmap="coolwarm", center=0, linewidths=0.2, cbar_kws={"shrink": 0.7})
plt.title("Mapa de correlacao entre variaveis")
plt.tight_layout()
plt.show()

Como esperado, as componentes `V1..V28` tem pouca correlacao entre si. Ainda assim, algumas delas apresentam associacao maior com `Class` e podem ter peso relevante nos modelos.

## 7. Pre-processamento

As colunas `V1..V28` ja vem transformadas pelo PCA. Ja `Time` e `Amount` estao em escalas bem diferentes, entao aplico `RobustScaler` nelas. Essa escolha e util porque `Amount` costuma ter valores extremos, e o `RobustScaler` e menos sensivel a outliers do que uma padronizacao comum.

In [ ]:
scaler = RobustScaler()

df_proc = df.copy()
df_proc[["Time", "Amount"]] = scaler.fit_transform(df_proc[["Time", "Amount"]])

print("Antes do scaling:")
display(df[["Time", "Amount"]].describe().loc[["mean", "std", "min", "max"]])

print("\nDepois do scaling (RobustScaler):")
display(df_proc[["Time", "Amount"]].describe().loc[["mean", "std", "min", "max"]])

## 8. Separacao entre treino e teste

Como a classe de fraude e muito pequena, a divisao entre treino e teste precisa preservar a proporcao das classes. Por isso uso `stratify=y` no `train_test_split`.

In [ ]:
X = df_proc.drop(columns=["Class"])
y = df_proc["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Treino: {X_train.shape[0]:,} amostras | Fraudes: {y_train.sum():,} ({y_train.mean()*100:.4f}%)")
print(f"Teste : {X_test.shape[0]:,} amostras | Fraudes: {y_test.sum():,} ({y_test.mean()*100:.4f}%)")

## 9. Estrategias para lidar com o desbalanceamento

Neste notebook comparo duas abordagens simples:

1. `class_weight="balanced"`: o proprio algoritmo aumenta o peso da classe minoritaria durante o treinamento.
2. `SMOTE`: cria exemplos sinteticos da classe minoritaria, usando apenas o conjunto de treino.

O ponto principal e nao aplicar oversampling antes da divisao treino/teste. Se isso fosse feito, informacoes do treino poderiam contaminar o teste e deixar as metricas artificialmente boas.

In [ ]:
if SMOTE_DISPONIVEL:
    smote = SMOTE(random_state=RANDOM_STATE)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

    print("Distribuicao da classe antes do SMOTE (somente treino):")
    print(y_train.value_counts())
    print("\nDistribuicao da classe depois do SMOTE (somente treino):")
    print(y_train_smote.value_counts())
else:
    X_train_smote, y_train_smote = X_train, y_train
    print("SMOTE indisponivel. O notebook seguira apenas com class_weight balanceado.")

## 10. Baseline com Regressao Logistica

A Regressao Logistica entra como modelo de referencia. Ela e simples, rapida de treinar e ajuda a entender se modelos mais complexos realmente trazem ganho.

In [ ]:
log_reg = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=RANDOM_STATE,
)
log_reg.fit(X_train, y_train)

y_pred_lr = log_reg.predict(X_test)
y_proba_lr = log_reg.predict_proba(X_test)[:, 1]

print("Relatorio de classificacao - Regressao Logistica (baseline):")
print(classification_report(y_test, y_pred_lr, target_names=["Legitima", "Fraude"], digits=4))

A matriz de confusao mostra os acertos e erros por classe. Para este problema, e especialmente importante observar os falsos negativos, que sao fraudes classificadas como transacoes legitimas.

In [ ]:
def plotar_matriz_confusao(y_true, y_pred, titulo):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt=",d", cmap="Blues",
        xticklabels=["Legitima", "Fraude"],
        yticklabels=["Legitima", "Fraude"],
    )
    plt.title(titulo)
    plt.ylabel("Classe real")
    plt.xlabel("Classe prevista")
    plt.tight_layout()
    plt.show()

plotar_matriz_confusao(y_test, y_pred_lr, "Matriz de confusao - Regressao Logistica")

Em bases muito desbalanceadas, a curva Precision-Recall costuma ser mais informativa que a curva ROC. A ROC pode parecer boa mesmo quando o modelo ainda tem dificuldade com a classe rara.

In [ ]:
def plotar_curvas(y_true, y_proba, nome_modelo, cor):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    fpr, tpr, _ = roc_curve(y_true, y_proba)
    auc_roc = roc_auc_score(y_true, y_proba)
    axes[0].plot(fpr, tpr, color=cor, label=f"{nome_modelo} (AUC = {auc_roc:.4f})")
    axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
    axes[0].set_title("Curva ROC")
    axes[0].set_xlabel("Taxa de falsos positivos")
    axes[0].set_ylabel("Taxa de verdadeiros positivos")
    axes[0].legend()

    precision, recall, _ = precision_recall_curve(y_true, y_proba)
    auprc = average_precision_score(y_true, y_proba)
    axes[1].plot(recall, precision, color=cor, label=f"{nome_modelo} (PR-AUC = {auprc:.4f})")
    axes[1].set_title("Curva Precision-Recall")
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].legend()

    plt.tight_layout()
    plt.show()
    return auc_roc, auprc

auc_roc_lr, auprc_lr = plotar_curvas(y_test, y_proba_lr, "Regressao Logistica", "#3498db")

## 11. Random Forest

Tambem treino uma Random Forest, que consegue capturar relacoes nao lineares e interacoes entre variaveis. Foram testadas duas versoes:

- Random Forest com `class_weight="balanced"`;
- Random Forest treinada com a base balanceada por `SMOTE`.

A comparacao entre as duas ajuda a verificar se o oversampling trouxe ganho neste caso.

In [ ]:
rf_balanced = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_balanced.fit(X_train, y_train)

y_pred_rf = rf_balanced.predict(X_test)
y_proba_rf = rf_balanced.predict_proba(X_test)[:, 1]

print("Relatorio de classificacao - Random Forest (class_weight balanceado):")
print(classification_report(y_test, y_pred_rf, target_names=["Legitima", "Fraude"], digits=4))

plotar_matriz_confusao(y_test, y_pred_rf, "Matriz de confusao - Random Forest (balanced)")
auc_roc_rf, auprc_rf = plotar_curvas(y_test, y_proba_rf, "Random Forest (balanced)", "#9b59b6")

In [ ]:
rf_smote = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_smote.fit(X_train_smote, y_train_smote)

y_pred_rf_smote = rf_smote.predict(X_test)
y_proba_rf_smote = rf_smote.predict_proba(X_test)[:, 1]

print("Relatorio de classificacao - Random Forest + SMOTE:")
print(classification_report(y_test, y_pred_rf_smote, target_names=["Legitima", "Fraude"], digits=4))

plotar_matriz_confusao(y_test, y_pred_rf_smote, "Matriz de confusao - Random Forest + SMOTE")
auc_roc_rf_smote, auprc_rf_smote = plotar_curvas(y_test, y_proba_rf_smote, "Random Forest + SMOTE", "#e67e22")

## 12. Comparacao dos modelos

Aqui reuno as principais metricas em uma tabela unica. A ordenacao usa PR-AUC, porque ela resume melhor o desempenho do modelo na classe rara do que a acuracia ou mesmo o ROC-AUC.

In [ ]:
resultados = pd.DataFrame({
    "Modelo": [
        "Regressao Logistica (balanced)",
        "Random Forest (balanced)",
        "Random Forest + SMOTE",
    ],
    "Precision": [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_rf_smote),
    ],
    "Recall": [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_rf_smote),
    ],
    "F1-score": [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_rf_smote),
    ],
    "ROC-AUC": [auc_roc_lr, auc_roc_rf, auc_roc_rf_smote],
    "PR-AUC": [auprc_lr, auprc_rf, auprc_rf_smote],
})

resultados = resultados.sort_values("PR-AUC", ascending=False).reset_index(drop=True)
display(resultados.style.background_gradient(cmap="Greens", subset=["Precision", "Recall", "F1-score", "ROC-AUC", "PR-AUC"]))

In [ ]:
metricas_plot = resultados.set_index("Modelo")[["Precision", "Recall", "F1-score", "PR-AUC"]]
metricas_plot.plot(kind="bar", figsize=(13, 6), colormap="viridis")
plt.title("Comparacao de metricas entre os modelos")
plt.ylabel("Score")
plt.xticks(rotation=15, ha="right")
plt.ylim(0, 1.05)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

melhor_modelo_nome = resultados.iloc[0]["Modelo"]
print(f"Melhor modelo por PR-AUC: {melhor_modelo_nome}")

Mesmo quando o ROC-AUC fica alto para mais de um modelo, a PR-AUC costuma mostrar melhor as diferencas na identificacao de fraudes. Por isso ela foi usada como criterio principal na comparacao.

## 13. Ajuste do threshold de decisao

O classificador binario usa, por padrao, threshold de 0,5. Esse valor nao precisa ser fixo. Em problemas de fraude, ele deve ser ajustado conforme o custo de cada tipo de erro.

- Threshold mais baixo: tende a aumentar o recall, mas tambem aumenta falsos positivos.
- Threshold mais alto: tende a aumentar a precision, mas pode deixar mais fraudes passarem.

Como nao ha uma matriz de custos definida para este exercicio, uso o F1-score como criterio simples para escolher um threshold de equilibrio.

In [ ]:
probabilidades_melhor_modelo = y_proba_rf_smote if "SMOTE" in melhor_modelo_nome else y_proba_rf

precisions, recalls, thresholds = precision_recall_curve(y_test, probabilidades_melhor_modelo)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-12)

idx_melhor_f1 = np.argmax(f1_scores)
melhor_threshold = thresholds[idx_melhor_f1] if idx_melhor_f1 < len(thresholds) else 0.5

print(f"Threshold que maximiza o F1-score: {melhor_threshold:.4f}")
print(f"Precision nesse ponto: {precisions[idx_melhor_f1]:.4f}")
print(f"Recall nesse ponto:    {recalls[idx_melhor_f1]:.4f}")
print(f"F1-score nesse ponto:  {f1_scores[idx_melhor_f1]:.4f}")

plt.figure(figsize=(11, 6))
plt.plot(thresholds, precisions[:-1], label="Precision", color="#3498db")
plt.plot(thresholds, recalls[:-1], label="Recall", color="#e74c3c")
plt.plot(thresholds, f1_scores[:-1], label="F1-score", color="#2ecc71", linestyle="--")
plt.axvline(melhor_threshold, color="black", linestyle=":", label=f"Threshold escolhido ({melhor_threshold:.3f})")
plt.xlabel("Threshold de decisao")
plt.ylabel("Score")
plt.title("Precision, recall e F1 por threshold")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
y_pred_threshold_ajustado = (probabilidades_melhor_modelo >= melhor_threshold).astype(int)

print("Relatorio de classificacao com threshold ajustado:")
print(classification_report(y_test, y_pred_threshold_ajustado, target_names=["Legitima", "Fraude"], digits=4))

plotar_matriz_confusao(y_test, y_pred_threshold_ajustado, f"Matriz de confusao - threshold ajustado ({melhor_threshold:.3f})")

## 14. Importancia das variaveis

Mesmo com variaveis anonimizadas, a Random Forest permite observar quais componentes tiveram maior peso nas decisoes do modelo. Isso nao explica diretamente o significado de cada variavel, mas ajuda a identificar quais componentes merecem atencao.

In [ ]:
importancias = pd.Series(rf_balanced.feature_importances_, index=X_train.columns)
top_15 = importancias.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 7))
sns.barplot(x=top_15.values, y=top_15.index, palette="rocket")
plt.title("Top 15 variaveis mais importantes - Random Forest")
plt.xlabel("Importancia (Gini)")
plt.tight_layout()
plt.show()

## 15. Conclusoes

A base tem um desbalanceamento muito forte: as fraudes representam cerca de 0,17% das transacoes. Por esse motivo, a acuracia nao e suficiente para avaliar os modelos. As metricas mais uteis aqui sao precision, recall, F1-score e principalmente PR-AUC.

Entre os modelos testados, a comparacao por PR-AUC indica qual abordagem lidou melhor com a classe de fraude. A Regressao Logistica funcionou como referencia inicial, enquanto a Random Forest permitiu testar um modelo mais flexivel, com e sem SMOTE.

O ajuste do threshold tambem e uma parte importante do problema. Em uma aplicacao real, essa escolha deveria considerar o custo de bloquear uma compra legitima e o custo de deixar passar uma fraude. Como esse custo nao foi informado, o F1-score foi usado como criterio de equilibrio.

## Pontos para um uso real

- Revalidar o modelo com dados mais recentes antes de qualquer uso em producao.
- Monitorar a queda de desempenho ao longo do tempo, ja que padroes de fraude mudam.
- Definir uma matriz de custos para escolher o threshold de forma mais alinhada ao negocio.
- Incluir outras informacoes, quando disponiveis, como historico do cliente, dispositivo, localizacao e frequencia de transacoes.
- Tratar casos intermediarios com revisao manual ou regras adicionais, em vez de depender apenas da decisao automatica.

## Limitacoes

As variaveis `V1..V28` sao componentes PCA anonimizadas. Isso protege informacoes sensiveis, mas reduz a interpretacao de negocio. Alem disso, a base original cobre apenas dois dias de transacoes de cartoes europeus em 2013, entao os resultados nao devem ser generalizados sem uma nova validacao.

Notebook feito para fins educacionais usando o dataset [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud), disponivel no Kaggle.